# Задание 4: Neural retrieval and re-ranking

Используем pre-trained dense models (bi-encoders и cross-encoders) для retrieval и re-ranking.

**Данные:**
- WikIR collection (en1k subset)
- MIRAGE collection

**План:**
1. **Ranking (25)** — retrieval с bi-encoder models
2. **Re-ranking (25)** — cross-encoder re-ranking top-k BM25
3. **Mixture model (30)** — оптимизация `alpha*BM25 + (1-alpha)*cosine`
4. **Additional — fine-tune cross-encoder (40)** — fine-tune cross-encoder на WikIR train

In [1]:
import json
import math
import pickle
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import ir_measures
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from ir_measures import AP, P, nDCG
from rank_bm25 import BM25Okapi
from scipy.optimize import minimize_scalar
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

/Users/maksimpiskaev/Проекты/IR/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
HW4_DIR = Path(".")
CACHE_DIR = HW4_DIR / "cache"
RUNS_DIR = HW4_DIR / "runs"
CACHE_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)

WIKIR_DIR = Path("../HW_1/data/wikIR1k")

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: mps


## Вспомогательные функции

In [3]:
TOKEN_RE = re.compile(r"[A-Za-z0-9_]+")

def simple_tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(str(text).lower())


def cache_path(name: str) -> Path:
    return CACHE_DIR / f"{name}.pkl"


def save_cache(name: str, obj):
    p = cache_path(name)
    with open(p, "wb") as f:
        pickle.dump(obj, f)
    print(f"  Cached: {p}")


def load_cache(name: str):
    p = cache_path(name)
    if p.exists():
        with open(p, "rb") as f:
            return pickle.load(f)
    return None


def evaluate_metrics(run_df: pd.DataFrame, qrels_df: pd.DataFrame) -> dict:
    """Вычисляет P@1, P@10, P@20, MAP, nDCG@20."""
    measures = [
        P(rel=1, judged_only=False) @ 1,
        P(rel=1, judged_only=False) @ 10,
        P(rel=1, judged_only=False) @ 20,
        AP(rel=1, judged_only=False),
        nDCG(dcg="log2", judged_only=False) @ 20,
    ]
    qrels_ir = [
        ir_measures.Qrel(str(q), str(d), int(r), "0")
        for q, d, r in qrels_df[["query_id", "doc_id", "relevance"]].itertuples(index=False)
    ]
    run_ir = [
        ir_measures.ScoredDoc(str(q), str(d), float(s))
        for q, d, s in run_df[["query_id", "doc_id", "score"]].itertuples(index=False)
    ]
    scores = ir_measures.calc_aggregate(measures, qrels_ir, run_ir)
    return {
        "P@1": scores[P(rel=1, judged_only=False) @ 1],
        "P@10": scores[P(rel=1, judged_only=False) @ 10],
        "P@20": scores[P(rel=1, judged_only=False) @ 20],
        "MAP": scores[AP(rel=1, judged_only=False)],
        "nDCG@20": scores[nDCG(dcg="log2", judged_only=False) @ 20],
    }


def build_run_df(query_ids, doc_ids, all_scores, run_id: str, top_k=100):
    """Строит run DataFrame."""
    rows = []
    for row_idx, qid in tqdm(enumerate(query_ids), total=len(query_ids), desc=f"build_run ({run_id})"):
        scores = all_scores[row_idx]
        valid_mask = scores > -np.inf
        if not valid_mask.any():
            continue
        valid_indices = np.where(valid_mask)[0]
        valid_scores = scores[valid_mask]
        top_order = np.argsort(valid_scores)[::-1][:top_k]
        for rank_pos, idx_in_valid in enumerate(top_order, start=1):
            doc_idx = valid_indices[idx_in_valid]
            rows.append({
                "query_id": str(qid),
                "doc_id": str(doc_ids[doc_idx]),
                "score": float(valid_scores[idx_in_valid]),
                "rank": rank_pos,
                "run_id": run_id,
            })
    return pd.DataFrame(rows)

### TextCollection — обёртка над коллекцией документов с BM25 и TF-IDF

In [4]:
class TextCollection:
    def __init__(self, docs_df, text_col="text", title_col=None):
        self.docs = docs_df.copy()
        self.docs["doc_id"] = self.docs["doc_id"].astype(str)
        self.doc_ids = self.docs["doc_id"].tolist()
        self.doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(self.doc_ids)}
        self.doc_tokens = [simple_tokenize(t) for t in tqdm(self.docs[text_col].fillna(""), desc="tokenize docs")]
        self.term_freqs = [Counter(t) for t in self.doc_tokens]
        self.bm25 = BM25Okapi(self.doc_tokens)
        text_strings = [" ".join(t) for t in self.doc_tokens]
        self.vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
        self.tfidf_matrix = self.vectorizer.fit_transform(text_strings)
        self.title_tokens = None
        self.title_bm25 = None
        if title_col is not None:
            self.title_tokens = [simple_tokenize(t) for t in tqdm(self.docs[title_col].fillna(""), desc="tokenize titles")]
            self.title_bm25 = BM25Okapi(self.title_tokens)

    def bm25_scores(self, query_text):
        return self.bm25.get_scores(simple_tokenize(query_text))

    def tfidf_scores(self, query_text):
        query = " ".join(simple_tokenize(query_text))
        return cosine_similarity(self.vectorizer.transform([query]), self.tfidf_matrix)[0]

    def title_bm25_scores(self, query_text):
        if self.title_bm25 is None:
            return None
        return self.title_bm25.get_scores(simple_tokenize(query_text))

## Загрузка данных

In [5]:
def load_wikir_split(split):
    documents = pd.read_csv(WIKIR_DIR / "documents.csv")
    documents.columns = ["doc_id", "text"]
    documents["doc_id"] = documents["doc_id"].astype(str)
    queries = pd.read_csv(WIKIR_DIR / split / "queries.csv")
    queries.columns = ["query_id", "text"]
    queries["query_id"] = queries["query_id"].astype(str)
    qrels = pd.read_csv(
        WIKIR_DIR / split / "qrels", sep="\t", header=None,
        names=["query_id", "iter", "doc_id", "relevance"],
    )
    qrels["query_id"] = qrels["query_id"].astype(str)
    qrels["doc_id"] = qrels["doc_id"].astype(str)
    return documents, queries, qrels


def load_mirage_data():
    """Загружает MIRAGE из HF с кешированием."""
    cached = load_cache("mirage_all")
    if cached is not None:
        docs_df, queries_df, qrels_df = cached
        print(f"  MIRAGE from cache: {len(docs_df)} docs, {len(queries_df)} queries")
        return docs_df, queries_df, qrels_df

    ds = load_dataset("nlpai-lab/mirage", cache_dir=str(CACHE_DIR / "hf_datasets"))["train"]
    docs = {}
    query_rows, qrels_rows = [], []
    for row in tqdm(ds, desc="MIRAGE parsing"):
        qid = str(row["query_id"])
        query_rows.append({"query_id": qid, "text": row["query"], "source": row["source"]})
        pool = row["doc_pool"]
        for mid, title, body, support in zip(
            pool["mapped_id"], pool["doc_name"], pool["doc_chunk"], pool["support"]
        ):
            key = (title, body)
            if key not in docs:
                docs[key] = {
                    "doc_id": f"M{len(docs)}", "title": title, "body": body,
                    "text": f"{title}\n{body}", "page_id": str(mid),
                }
            qrels_rows.append({
                "query_id": qid, "iter": 0,
                "doc_id": docs[key]["doc_id"], "relevance": int(support),
            })
    docs_df = pd.DataFrame(docs.values())
    queries_df = pd.DataFrame(query_rows).drop_duplicates("query_id")
    qrels_df = pd.DataFrame(qrels_rows)
    save_cache("mirage_all", (docs_df, queries_df, qrels_df))
    return docs_df, queries_df, qrels_df


def prepare_mirage_splits(queries_df, qrels_df):
    qids = np.array(list(queries_df["query_id"]))
    sources = np.array(list(queries_df["source"]))
    train_qids, test_qids = train_test_split(qids, test_size=0.2, random_state=42, stratify=sources)
    train_qids, test_qids = set(map(str, train_qids)), set(map(str, test_qids))
    train_q = queries_df[queries_df["query_id"].isin(train_qids)].copy()
    test_q = queries_df[queries_df["query_id"].isin(test_qids)].copy()
    train_qr = qrels_df[qrels_df["query_id"].isin(train_qids)].copy()
    test_qr = qrels_df[qrels_df["query_id"].isin(test_qids)].copy()
    return train_q, test_q, train_qr, test_qr

In [6]:
# WikIR
wikir_documents, wikir_test_queries, wikir_test_qrels = load_wikir_split("test")
_, wikir_train_queries, wikir_train_qrels = load_wikir_split("training")
print(f"WikIR docs: {len(wikir_documents):,}")
print(f"WikIR train queries: {len(wikir_train_queries):,}")
print(f"WikIR test queries: {len(wikir_test_queries):,}")

WikIR docs: 369,721
WikIR train queries: 1,444
WikIR test queries: 100


In [7]:
# MIRAGE
mirage_documents, mirage_queries_all, mirage_qrels_all = load_mirage_data()
mirage_train_queries, mirage_test_queries, mirage_train_qrels, mirage_test_qrels = prepare_mirage_splits(
    mirage_queries_all, mirage_qrels_all
)
print(f"MIRAGE docs: {len(mirage_documents):,}")
print(f"MIRAGE train: {len(mirage_train_queries)}, test: {len(mirage_test_queries)}")

  MIRAGE from cache: 36950 docs, 7560 queries
MIRAGE docs: 36,950
MIRAGE train: 6048, test: 1512


In [8]:
# TextCollections
wikir_coll = load_cache("wikir_collection")
if wikir_coll is None:
    wikir_coll = TextCollection(wikir_documents)
    save_cache("wikir_collection", wikir_coll)

mirage_coll = load_cache("mirage_collection")
if mirage_coll is None:
    mirage_coll = TextCollection(mirage_documents, title_col="title")
    save_cache("mirage_collection", mirage_coll)

---

## Task 1: Ranking с bi-encoder models (25 points)

Используем две модели:

1. **`all-MiniLM-L6-v2`** — быстрая, 80M параметров, хороший baseline
2. **`all-mpnet-base-v2`** — более глубокая, одна из лучших general-purpose bi-encoder

In [9]:
BI_ENCODER_MODELS = {
    "minilm_l6": "all-MiniLM-L6-v2",
    "mpnet_base": "all-mpnet-base-v2",
}

In [10]:
def encode_and_retrieve(coll, queries_df, short_name, model, top_k=200, batch_size=64):
    """Dense retrieval: encode docs + queries, compute cosine similarity."""
    model_name = short_name.lower()
    effective_batch_size = batch_size
    if "mpnet" in model_name:
        # На этой конфигурации MPS с bs=64 быстрее, чем bs=32/CPU.
        effective_batch_size = min(batch_size, 64)
    elif "minilm" in model_name:
        effective_batch_size = max(batch_size, 128)
    print(f"  batch_size={effective_batch_size}")

    # Encode documents (с кешированием и возобновлением по чанкам)
    cache_key = f"{short_name}_doc_emb_{len(coll.doc_ids)}"
    doc_emb = load_cache(cache_key)
    if doc_emb is None:
        doc_texts = coll.docs["text"].fillna("").tolist()
        chunk_docs = 50_000
        chunk_arrays = []
        n_chunks = (len(doc_texts) + chunk_docs - 1) // chunk_docs
        print(f"  encoding docs in {n_chunks} chunks x <= {chunk_docs} docs")

        for ci in tqdm(range(n_chunks), desc=f"doc chunks ({short_name})"):
            chunk_key = f"{cache_key}_chunk{ci}"
            arr = load_cache(chunk_key)
            if arr is None:
                start = ci * chunk_docs
                end = min((ci + 1) * chunk_docs, len(doc_texts))
                emb_chunk = model.encode(
                    doc_texts[start:end],
                    batch_size=effective_batch_size,
                    show_progress_bar=True,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                )
                arr = emb_chunk.astype(np.float16, copy=False)
                save_cache(chunk_key, arr)
            chunk_arrays.append(arr.astype(np.float32, copy=False))

        doc_emb = np.concatenate(chunk_arrays, axis=0)
        save_cache(cache_key, doc_emb.astype(np.float16))
    else:
        print("  doc embeddings from cache")
        if isinstance(doc_emb, torch.Tensor):
            doc_emb = doc_emb.detach().cpu().numpy()
        doc_emb = doc_emb.astype(np.float32, copy=False)

    # Encode queries
    q_texts = queries_df["text"].tolist()
    q_emb = model.encode(
        q_texts,
        batch_size=effective_batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32, copy=False)

    # Cosine similarity (эмбеддинги уже L2-нормированы => скалярное произведение)
    sim_matrix = q_emb @ doc_emb.T

    # Leave only top-k per query
    n_queries, n_docs = sim_matrix.shape
    result_matrix = np.full((n_queries, n_docs), -np.inf, dtype=np.float32)
    for qi in tqdm(range(n_queries), desc=f"select top-{top_k} ({short_name})"):
        sims = sim_matrix[qi]
        if top_k >= n_docs:
            top_idx = np.argsort(sims)[::-1]
        else:
            top_idx = np.argpartition(sims, -top_k)[-top_k:]
            top_idx = top_idx[np.argsort(sims[top_idx])[::-1]]
        result_matrix[qi, top_idx] = sims[top_idx]

    qids = queries_df["query_id"].tolist()
    return build_run_df(qids, coll.doc_ids, result_matrix, f"dense_{short_name}", top_k=top_k)


In [11]:
# Загрузка моделей
bi_encoder_models = {}
for short_name, full_name in tqdm(BI_ENCODER_MODELS.items(), desc="Loading bi-encoders"):
    print(f"  {short_name}: {full_name}")
    bi_encoder_models[short_name] = SentenceTransformer(full_name, device=device)

    # На Apple MPS fp16 заметно ускоряет mpnet без существенной просадки качества.
    if device == "mps" and "mpnet" in short_name:
        bi_encoder_models[short_name].half()
        print("    -> switched to fp16 for faster MPS inference")

    print(f"    -> {bi_encoder_models[short_name].get_sentence_embedding_dimension()} dim, device={bi_encoder_models[short_name].device}")


Loading bi-encoders:   0%|          | 0/2 [00:00<?, ?it/s]

  minilm_l6: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6847.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading bi-encoders:  50%|█████     | 1/2 [00:13<00:13, 13.97s/it]

    -> 384 dim, device=mps:0
  mpnet_base: all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6233.97it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading bi-encoders: 100%|██████████| 2/2 [00:29<00:00, 14.67s/it]

    -> switched to fp16 for faster MPS inference
    -> 768 dim, device=mps:0


In [12]:
# Task 1: WikIR test
all_results = {}

for short_name, model in tqdm(bi_encoder_models.items(), desc="Task1 WikIR"):
    print(f"\n--- Dense retrieval: {short_name} (WikIR) ---")
    wikir_run = encode_and_retrieve(wikir_coll, wikir_test_queries, short_name, model)
    wikir_metrics = evaluate_metrics(wikir_run, wikir_test_qrels)
    all_results[f"dense_{short_name}_wikir"] = {"run": wikir_run, "metrics": wikir_metrics}
    print(f"  {wikir_metrics}")

Task1 WikIR:   0%|          | 0/2 [00:00<?, ?it/s]


--- Dense retrieval: minilm_l6 (WikIR) ---
  batch_size=128
  doc embeddings from cache


Task1 WikIR:  50%|█████     | 1/2 [00:02<00:02,  2.08s/it]

  {'P@1': 0.64, 'P@10': 0.18499999999999994, 'P@20': 0.12650000000000003, 'MAP': 0.13725967865781538, 'nDCG@20': 0.3574088832517397}

--- Dense retrieval: mpnet_base (WikIR) ---
  batch_size=64
  doc embeddings from cache


Task1 WikIR: 100%|██████████| 2/2 [00:03<00:00,  1.78s/it]

  {'P@1': 0.78, 'P@10': 0.20799999999999996, 'P@20': 0.136, 'MAP': 0.16801269805146812, 'nDCG@20': 0.40213498121235786}


In [13]:
# Task 1: MIRAGE test
for short_name, model in tqdm(bi_encoder_models.items(), desc="Task1 MIRAGE"):
    print(f"\n--- Dense retrieval: {short_name} (MIRAGE) ---")
    mirage_run = encode_and_retrieve(mirage_coll, mirage_test_queries, f"mirage_{short_name}", model)
    mirage_metrics = evaluate_metrics(mirage_run, mirage_test_qrels)
    all_results[f"dense_{short_name}_mirage"] = {"run": mirage_run, "metrics": mirage_metrics}
    print(f"  {mirage_metrics}")

Task1 MIRAGE:   0%|          | 0/2 [00:00<?, ?it/s]


--- Dense retrieval: minilm_l6 (MIRAGE) ---
  batch_size=128
  doc embeddings from cache


Task1 MIRAGE:  50%|█████     | 1/2 [00:08<00:08,  8.69s/it]

  {'P@1': 0.583994708994709, 'P@10': 0.11130952380952118, 'P@20': 0.05753968253968113, 'MAP': 0.706316823807933, 'nDCG@20': 0.7710260581258458}

--- Dense retrieval: mpnet_base (MIRAGE) ---
  batch_size=64
  doc embeddings from cache


Task1 MIRAGE: 100%|██████████| 2/2 [00:21<00:00, 10.91s/it]

  {'P@1': 0.5548941798941799, 'P@10': 0.11025132275132024, 'P@20': 0.057142857142855774, 'MAP': 0.6851008746236513, 'nDCG@20': 0.7527855336373556}


In [14]:
# Task 1: Сводная таблица
print("TASK 1: Dense Retrieval Results")
print(f"{'Model':<20s} {'Dataset':<10s} {'P@1':>8s} {'P@10':>8s} {'P@20':>8s} {'MAP':>8s} {'nDCG@20':>8s}")
print("-" * 80)
for key in sorted(all_results.keys()):
    if key.startswith("dense_"):
        parts = key.split("_")
        model_name = parts[1] + "_" + parts[2] if len(parts) > 3 else parts[1]
        dataset = key.split("_")[-1]
        m = all_results[key]["metrics"]
        print(f"{model_name:<20s} {dataset:<10s} {m['P@1']:>8.4f} {m['P@10']:>8.4f} {m['P@20']:>8.4f} {m['MAP']:>8.4f} {m['nDCG@20']:>8.4f}")

TASK 1: Dense Retrieval Results
Model                Dataset         P@1     P@10     P@20      MAP  nDCG@20
--------------------------------------------------------------------------------
minilm_l6            mirage       0.5840   0.1113   0.0575   0.7063   0.7710
minilm_l6            wikir        0.6400   0.1850   0.1265   0.1373   0.3574
mpnet_base           mirage       0.5549   0.1103   0.0571   0.6851   0.7528
mpnet_base           wikir        0.7800   0.2080   0.1360   0.1680   0.4021


---

## Task 2: Re-ranking с cross-encoder (25 points)

Re-rank top-k BM25 результатов с помощью cross-encoder `ms-marco-MiniLM-L-6-v2`.
Эксперименты с k ∈ {10, 30, 50, 100}.

In [15]:
CE_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print(f"Loading cross-encoder: {CE_MODEL_NAME}")
cross_encoder = CrossEncoder(CE_MODEL_NAME, device=device)

K_VALUES = [10, 30, 50, 100]

Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6782.51it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def bm25_crossencoder_rerank(coll, queries_df, ce_model, top_k, batch_size=64, run_id_prefix="ce"):
    """Re-rank top-k BM25 результатов через cross-encoder."""
    qids = queries_df["query_id"].tolist()
    qtexts = queries_df["text"].tolist()
    doc_ids = coll.doc_ids
    doc_texts = coll.docs["text"].fillna("").tolist()
    rows = []

    for q_idx, (qid, qtext) in tqdm(enumerate(zip(qids, qtexts)), total=len(qids), desc=f"CE rerank k={top_k}"):
        bm25_scores = coll.bm25_scores(qtext)
        top_indices = np.argsort(bm25_scores)[::-1][:top_k]
        pairs = [(qtext, doc_texts[idx]) for idx in top_indices]
        if not pairs:
            continue
        ce_scores = ce_model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        for rank_pos, doc_idx in enumerate(top_indices, start=1):
            rows.append({
                "query_id": str(qid),
                "doc_id": str(doc_ids[doc_idx]),
                "score": float(ce_scores[rank_pos - 1]),
                "rank": rank_pos,
                "run_id": f"{run_id_prefix}_k{top_k}",
            })
    return pd.DataFrame(rows)

In [17]:

# Task 2: WikIR
for k in tqdm(K_VALUES, desc="WikIR CE rerank"):
    cache_key = f"ce_rerank_wikir_k{k}"
    cached = load_cache(cache_key)
    if cached is not None:
        run_df, metrics = cached
        print(f"  k={k}: from cache — {metrics}")
    else:
        run_df = bm25_crossencoder_rerank(wikir_coll, wikir_test_queries, cross_encoder, k, run_id_prefix="wikir_ce")
        metrics = evaluate_metrics(run_df, wikir_test_qrels)
        save_cache(cache_key, (run_df, metrics))
        print(f"  k={k}: {metrics}")
    all_results[f"ce_k{k}_wikir"] = {"run": run_df, "metrics": metrics}


WikIR CE rerank: 100%|██████████| 4/4 [00:00<00:00, 270.78it/s]

  k=10: from cache — {'P@1': 0.74, 'P@10': 0.207, 'P@20': 0.1035, 'MAP': 0.15109242913729826, 'nDCG@20': 0.3696416176757129}
  k=30: from cache — {'P@1': 0.8, 'P@10': 0.22799999999999998, 'P@20': 0.16100000000000006, 'MAP': 0.18621281471469936, 'nDCG@20': 0.43906759232100967}
  k=50: from cache — {'P@1': 0.81, 'P@10': 0.231, 'P@20': 0.163, 'MAP': 0.19815079867280028, 'nDCG@20': 0.4461974458470827}
  k=100: from cache — {'P@1': 0.8, 'P@10': 0.23200000000000004, 'P@20': 0.16699999999999998, 'MAP': 0.20503645385349573, 'nDCG@20': 0.4491509743505272}


In [18]:
# Task 2: MIRAGE
for k in tqdm(K_VALUES, desc="MIRAGE CE rerank"):
    cache_key = f"ce_rerank_mirage_k{k}"
    cached = load_cache(cache_key)
    if cached is not None:
        run_df, metrics = cached
        print(f"  k={k}: from cache — {metrics}")
    else:
        run_df = bm25_crossencoder_rerank(mirage_coll, mirage_test_queries, cross_encoder, k, run_id_prefix="mirage_ce")
        metrics = evaluate_metrics(run_df, mirage_test_qrels)
        save_cache(cache_key, (run_df, metrics))
        print(f"  k={k}: {metrics}")
    all_results[f"ce_k{k}_mirage"] = {"run": run_df, "metrics": metrics}


MIRAGE CE rerank: 100%|██████████| 4/4 [00:00<00:00, 77.85it/s]

  k=10: from cache — {'P@1': 0.6937830687830688, 'P@10': 0.09742063492063294, 'P@20': 0.04871031746031647, 'MAP': 0.7336418440833131, 'nDCG@20': 0.759604639465114}
  k=30: from cache — {'P@1': 0.7321428571428571, 'P@10': 0.10562169312169073, 'P@20': 0.05297619047618928, 'MAP': 0.7826687570586388, 'nDCG@20': 0.8119069462956036}
  k=50: from cache — {'P@1': 0.7420634920634921, 'P@10': 0.10800264550264298, 'P@20': 0.05410052910052784, 'MAP': 0.7954831719948674, 'nDCG@20': 0.8256611363758622}
  k=100: from cache — {'P@1': 0.7506613756613757, 'P@10': 0.11031746031745772, 'P@20': 0.055390211640210324, 'MAP': 0.8084749885854385, 'nDCG@20': 0.840366997795488}


In [19]:

# Task 2: Визуализация зависимости метрик от k
# all_results уже заполнен из кеша в ячейках выше
ce_wikir_data = {k: all_results[f"ce_k{k}_wikir"]["metrics"] for k in K_VALUES}
ce_mirage_data = {k: all_results[f"ce_k{k}_mirage"]["metrics"] for k in K_VALUES}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for split_name, data, ax in [("WikIR", ce_wikir_data, axes[0]), ("MIRAGE", ce_mirage_data, axes[1])]:
    for metric in ["P@1", "P@10", "MAP", "nDCG@20"]:
        vals = [data[k][metric] for k in K_VALUES]
        ax.plot(K_VALUES, vals, marker="o", label=metric)
    ax.set_title(f"CE re-rank: {split_name}")
    ax.set_xlabel("k (BM25 top-k)")
    ax.set_ylabel("Metric")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(HW4_DIR / "ce_rerank_k_analysis.png", dpi=150)
plt.close()
print("Saved: ce_rerank_k_analysis.png")


Saved: ce_rerank_k_analysis.png


---

## Task 3: Mixture model — alpha*BM25 + (1-alpha)*cosine (30 points)


BM25 нормализуется min-max в [0, 1] для каждого запроса.

In [20]:
def minmax_normalize(arr):
    arr = np.array(arr, dtype=np.float64)
    mn, mx = arr.min(), arr.max()
    if mx == mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)


def evaluate_mixture(coll, queries_df, qrels_df, bi_model, alpha, top_k=100, batch_size=64, desc="mixture"):
    """Оценка mixture model для заданного alpha."""
    qids = queries_df["query_id"].tolist()
    qtexts = queries_df["text"].tolist()
    doc_ids = coll.doc_ids
    doc_texts = coll.docs["text"].fillna("").tolist()

    doc_emb = bi_model.encode(doc_texts, batch_size=batch_size, show_progress_bar=True,
                               convert_to_tensor=True, normalize_embeddings=True)

    rows = []
    for q_idx, (qid, qtext) in tqdm(enumerate(zip(qids, qtexts)), total=len(qids), desc=f"{desc} a={alpha:.2f}"):
        bm25_scores = coll.bm25_scores(qtext)
        bm25_norm = minmax_normalize(bm25_scores)
        q_emb = bi_model.encode([qtext], convert_to_tensor=True, normalize_embeddings=True)
        cos_sims = util.cos_sim(q_emb, doc_emb)[0].cpu().numpy()
        mixed = alpha * bm25_norm + (1 - alpha) * cos_sims
        top_indices = np.argsort(mixed)[::-1][:top_k]
        for rank_pos, doc_idx in enumerate(top_indices, start=1):
            rows.append({
                "query_id": str(qid), "doc_id": str(doc_ids[doc_idx]),
                "score": float(mixed[doc_idx]), "rank": rank_pos,
                "run_id": f"mixture_a{alpha:.2f}",
            })
    run_df = pd.DataFrame(rows)
    return evaluate_metrics(run_df, qrels_df)

In [21]:
# --- Оптимизация alpha на WikIR train (быстрая версия) ---
# Проблема оригинала: evaluate_mixture заново кодирует 370K документов при каждом вызове (~1.5ч/итерация).
# Решение: предвычислить всё один раз, оптимизация alpha = просто векторное сложение.

minilm = bi_encoder_models["minilm_l6"]
train_q_texts = wikir_train_queries["text"].tolist()
train_qids = wikir_train_queries["query_id"].tolist()

# 1. Эмбеддинги документов — уже в кеше
print("Загрузка эмбеддингов документов...")
_doc_emb_raw = load_cache("minilm_l6_doc_emb_369721")
if isinstance(_doc_emb_raw, torch.Tensor):
    _doc_emb_raw = _doc_emb_raw.detach().cpu().numpy()
doc_emb_train = _doc_emb_raw.astype(np.float32)
print(f"  doc_emb shape: {doc_emb_train.shape}")

# 2. Эмбеддинги тренировочных запросов
q_emb_cache_key = f"minilm_l6_train_q_emb_{len(train_q_texts)}"
q_emb_train = load_cache(q_emb_cache_key)
if q_emb_train is None:
    print("Кодирование train запросов...")
    q_emb_train = minilm.encode(
        train_q_texts, batch_size=256, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32)
    save_cache(q_emb_cache_key, q_emb_train)
else:
    if isinstance(q_emb_train, torch.Tensor):
        q_emb_train = q_emb_train.detach().cpu().numpy()
    q_emb_train = q_emb_train.astype(np.float32)
    print("  train query embeddings from cache")
print(f"  q_emb shape: {q_emb_train.shape}")

# 3. Матрица косинусных сходств
cos_cache_key = f"minilm_l6_wikir_train_cos_matrix_{len(train_qids)}x{len(wikir_coll.doc_ids)}"
cos_matrix = load_cache(cos_cache_key)
if cos_matrix is None:
    print("Вычисление cosine similarity матрицы...")
    cos_matrix = (q_emb_train @ doc_emb_train.T).astype(np.float32)
    save_cache(cos_cache_key, cos_matrix)
else:
    if isinstance(cos_matrix, torch.Tensor):
        cos_matrix = cos_matrix.detach().cpu().numpy()
    cos_matrix = cos_matrix.astype(np.float32)
    print("  cos_matrix from cache")
print(f"  cos_matrix shape: {cos_matrix.shape}")

# 4. BM25 скоры для всех тренировочных запросов
bm25_cache_key = f"wikir_train_bm25_matrix_{len(train_qids)}x{len(wikir_coll.doc_ids)}"
bm25_matrix = load_cache(bm25_cache_key)
if bm25_matrix is None:
    print("Вычисление BM25 скоров для train запросов...")
    bm25_matrix = np.zeros_like(cos_matrix)
    for i, qtext in tqdm(enumerate(train_q_texts), total=len(train_q_texts), desc="BM25 train"):
        raw = wikir_coll.bm25_scores(qtext)
        mn, mx = raw.min(), raw.max()
        bm25_matrix[i] = (raw - mn) / (mx - mn) if mx > mn else raw
    save_cache(bm25_cache_key, bm25_matrix)
else:
    if isinstance(bm25_matrix, torch.Tensor):
        bm25_matrix = bm25_matrix.detach().cpu().numpy()
    bm25_matrix = bm25_matrix.astype(np.float32)
    print("  bm25_matrix from cache")
print(f"  bm25_matrix shape: {bm25_matrix.shape}")

print("Всё предвычислено. Запуск оптимизации alpha...")

def fast_objective(alpha):
    """Смешиваем предвычисленные скоры — никакого inference."""
    mixed = alpha * bm25_matrix + (1.0 - alpha) * cos_matrix  # (n_queries, n_docs)
    top_k = 100
    rows = []
    for qi, qid in enumerate(train_qids):
        row_scores = mixed[qi]
        top_idx = np.argpartition(row_scores, -top_k)[-top_k:]
        top_idx = top_idx[np.argsort(row_scores[top_idx])[::-1]]
        for rank, idx in enumerate(top_idx, start=1):
            rows.append({
                "query_id": str(qid),
                "doc_id": str(wikir_coll.doc_ids[idx]),
                "score": float(row_scores[idx]),
                "rank": rank,
                "run_id": f"mix_a{alpha:.2f}",
            })
    run_df = pd.DataFrame(rows)
    m = evaluate_metrics(run_df, wikir_train_qrels)
    print(f"  alpha={alpha:.4f}  nDCG@20={m['nDCG@20']:.4f}")
    return -m["nDCG@20"]

result = minimize_scalar(fast_objective, bounds=(0, 1), method="bounded", options={"xatol": 0.01})
optimal_alpha = result.x
print(f"\nOptimal alpha: {optimal_alpha:.3f}")
optimal_ndcg = -result.fun
print(f"Train nDCG@20 at optimal alpha: {optimal_ndcg:.4f}")


Загрузка эмбеддингов документов...
  doc_emb shape: (369721, 384)
  train query embeddings from cache
  q_emb shape: (1444, 384)
  cos_matrix from cache
  cos_matrix shape: (1444, 369721)
  bm25_matrix from cache
  bm25_matrix shape: (1444, 369721)
Всё предвычислено. Запуск оптимизации alpha...
  alpha=0.3820  nDCG@20=0.4299
  alpha=0.6180  nDCG@20=0.4105
  alpha=0.2361  nDCG@20=0.4315
  alpha=0.2806  nDCG@20=0.4314
  alpha=0.2433  nDCG@20=0.4316
  alpha=0.2546  nDCG@20=0.4311
  alpha=0.2476  nDCG@20=0.4314
  alpha=0.2400  nDCG@20=0.4312

Optimal alpha: 0.243
Train nDCG@20 at optimal alpha: 0.4316


In [22]:
cache_key_wikir_test = f"wikir_test_mixture_a{optimal_alpha:.3f}"
cached_mix = load_cache(cache_key_wikir_test)
if cached_mix is not None:
    wikir_mix_metrics = cached_mix
    print(f"  from cache: {wikir_mix_metrics}")
else:
    print(f"Применение alpha={optimal_alpha:.3f} на WikIR test (fast)...")
    test_q_texts = wikir_test_queries["text"].tolist()
    test_qids = wikir_test_queries["query_id"].tolist()

    # Загружаем кешированные doc-эмбеддинги
    _doc_emb = load_cache("minilm_l6_doc_emb_369721")
    if isinstance(_doc_emb, torch.Tensor):
        _doc_emb = _doc_emb.detach().cpu().numpy()
    doc_emb_wtest = _doc_emb.astype(np.float32)

    # Кодируем test-запросы
    q_emb_wtest_key = f"minilm_l6_wikir_test_q_emb_{len(test_q_texts)}"
    q_emb_wtest = load_cache(q_emb_wtest_key)
    if q_emb_wtest is None:
        q_emb_wtest = minilm.encode(
            test_q_texts, batch_size=128, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        ).astype(np.float32)
        save_cache(q_emb_wtest_key, q_emb_wtest)
    else:
        q_emb_wtest = q_emb_wtest.astype(np.float32)
        print("  test query embeddings from cache")

    cos_wtest = (q_emb_wtest @ doc_emb_wtest.T).astype(np.float32)

    bm25_wtest = np.zeros_like(cos_wtest)
    for i, qtext in tqdm(enumerate(test_q_texts), total=len(test_q_texts), desc="BM25 WikIR test"):
        raw = wikir_coll.bm25_scores(qtext)
        mn, mx = raw.min(), raw.max()
        bm25_wtest[i] = (raw - mn) / (mx - mn) if mx > mn else raw

    top_k = 100
    rows = []
    for qi, qid in enumerate(test_qids):
        mixed = optimal_alpha * bm25_wtest[qi] + (1.0 - optimal_alpha) * cos_wtest[qi]
        top_idx = np.argpartition(mixed, -top_k)[-top_k:]
        top_idx = top_idx[np.argsort(mixed[top_idx])[::-1]]
        for rank, idx in enumerate(top_idx, start=1):
            rows.append({
                "query_id": str(qid), "doc_id": str(wikir_coll.doc_ids[idx]),
                "score": float(mixed[idx]), "rank": rank,
                "run_id": f"mixture_a{optimal_alpha:.2f}",
            })
    wikir_mix_metrics = evaluate_metrics(pd.DataFrame(rows), wikir_test_qrels)
    save_cache(cache_key_wikir_test, wikir_mix_metrics)

all_results[f"mixture_a{optimal_alpha:.2f}_wikir"] = {"metrics": wikir_mix_metrics}
print(f"  WikIR test mixture: {wikir_mix_metrics}")


  from cache: {'P@1': 0.78, 'P@10': 0.243, 'P@20': 0.168, 'MAP': 0.202723037970463, 'nDCG@20': 0.44199225908316575}
  WikIR test mixture: {'P@1': 0.78, 'P@10': 0.243, 'P@20': 0.168, 'MAP': 0.202723037970463, 'nDCG@20': 0.44199225908316575}


In [23]:
# --- Применение optimal alpha на MIRAGE test (быстрая версия) ---
cache_key_mirage_test = f"mirage_test_mixture_a{optimal_alpha:.3f}"
cached_mix_m = load_cache(cache_key_mirage_test)
if cached_mix_m is not None:
    mirage_mix_metrics = cached_mix_m
    print(f"  from cache: {mirage_mix_metrics}")
else:
    print(f"Применение alpha={optimal_alpha:.3f} на MIRAGE test (fast)...")
    mtest_q_texts = mirage_test_queries["text"].tolist()
    mtest_qids = mirage_test_queries["query_id"].tolist()

    # MIRAGE: 37K документов — кодируем в одном проходе
    _mdoc_emb_key = f"mirage_minilm_l6_doc_emb_{len(mirage_coll.doc_ids)}"
    mdoc_emb = load_cache(_mdoc_emb_key)
    if mdoc_emb is None:
        mdoc_emb = minilm.encode(
            mirage_coll.docs["text"].fillna("").tolist(),
            batch_size=128, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        ).astype(np.float32)
        save_cache(_mdoc_emb_key, mdoc_emb)
    else:
        mdoc_emb = mdoc_emb.astype(np.float32)
        print("  MIRAGE doc embeddings from cache")

    mq_emb_key = f"minilm_l6_mirage_test_q_emb_{len(mtest_q_texts)}"
    mq_emb = load_cache(mq_emb_key)
    if mq_emb is None:
        mq_emb = minilm.encode(
            mtest_q_texts, batch_size=128, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        ).astype(np.float32)
        save_cache(mq_emb_key, mq_emb)
    else:
        mq_emb = mq_emb.astype(np.float32)
        print("  MIRAGE test query embeddings from cache")

    cos_mtest = (mq_emb @ mdoc_emb.T).astype(np.float32)

    bm25_mtest = np.zeros_like(cos_mtest)
    for i, qtext in tqdm(enumerate(mtest_q_texts), total=len(mtest_q_texts), desc="BM25 MIRAGE test"):
        raw = mirage_coll.bm25_scores(qtext)
        mn, mx = raw.min(), raw.max()
        bm25_mtest[i] = (raw - mn) / (mx - mn) if mx > mn else raw

    top_k = 100
    rows = []
    for qi, qid in enumerate(mtest_qids):
        mixed = optimal_alpha * bm25_mtest[qi] + (1.0 - optimal_alpha) * cos_mtest[qi]
        top_idx = np.argpartition(mixed, -top_k)[-top_k:]
        top_idx = top_idx[np.argsort(mixed[top_idx])[::-1]]
        for rank, idx in enumerate(top_idx, start=1):
            rows.append({
                "query_id": str(qid), "doc_id": str(mirage_coll.doc_ids[idx]),
                "score": float(mixed[idx]), "rank": rank,
                "run_id": f"mixture_a{optimal_alpha:.2f}",
            })
    mirage_mix_metrics = evaluate_metrics(pd.DataFrame(rows), mirage_test_qrels)
    save_cache(cache_key_mirage_test, mirage_mix_metrics)

all_results[f"mixture_a{optimal_alpha:.2f}_mirage"] = {"metrics": mirage_mix_metrics}
print(f"  MIRAGE test mixture: {mirage_mix_metrics}")


  from cache: {'P@1': 0.658068783068783, 'P@10': 0.11547619047618764, 'P@20': 0.059027777777776284, 'MAP': 0.7652598010707944, 'nDCG@20': 0.820698935454394}
  MIRAGE test mixture: {'P@1': 0.658068783068783, 'P@10': 0.11547619047618764, 'P@20': 0.059027777777776284, 'MAP': 0.7652598010707944, 'nDCG@20': 0.820698935454394}


## Сводная таблица всех подходов


In [24]:
# Собираем summary
summary_rows = []

# Dense retrieval
for key in sorted(all_results):
    if key.startswith("dense_"):
        parts = key.replace("dense_", "").split("_")
        model = parts[0] + "_" + parts[1] if len(parts) > 2 else parts[0]
        dataset = key.split("_")[-1]
        m = all_results[key]["metrics"]
        summary_rows.append({"Система": f"Dense {model}", "Dataset": dataset, **m})

# CE re-rank (best k)
for dataset in ["wikir", "mirage"]:
    best_key = None
    best_ndcg = -1
    for k in K_VALUES:
        key = f"ce_k{k}_{dataset}"
        if key in all_results:
            ndcg = all_results[key]["metrics"]["nDCG@20"]
            if ndcg > best_ndcg:
                best_ndcg = ndcg
                best_key = key
    if best_key:
        m = all_results[best_key]["metrics"]
        summary_rows.append({"Система": f"CE rerank (k={best_key.split('_')[1][1:]})", "Dataset": dataset.upper(), **m})

# Mixture
for dataset in ["wikir", "mirage"]:
    key = f"mixture_a{optimal_alpha:.2f}_{dataset}"
    if key in all_results:
        m = all_results[key]["metrics"]
        summary_rows.append({"Система": f"Mixture (α={optimal_alpha:.2f})", "Dataset": dataset.upper(), **m})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

          Система Dataset      P@1     P@10     P@20      MAP  nDCG@20
  Dense minilm_l6  mirage 0.583995 0.111310 0.057540 0.706317 0.771026
  Dense minilm_l6   wikir 0.640000 0.185000 0.126500 0.137260 0.357409
 Dense mpnet_base  mirage 0.554894 0.110251 0.057143 0.685101 0.752786
 Dense mpnet_base   wikir 0.780000 0.208000 0.136000 0.168013 0.402135
CE rerank (k=100)   WIKIR 0.800000 0.232000 0.167000 0.205036 0.449151
CE rerank (k=100)  MIRAGE 0.750661 0.110317 0.055390 0.808475 0.840367
 Mixture (α=0.24)   WIKIR 0.780000 0.243000 0.168000 0.202723 0.441992
 Mixture (α=0.24)  MIRAGE 0.658069 0.115476 0.059028 0.765260 0.820699


---

## Additional Task: Fine-tune cross-encoder (40 points)

Fine-tune cross-encoder на WikIR training data:
- **Positive pairs:** (query, relevant_doc) из qrels (relevance >= 1)
- **Negative pairs:** hard negatives из BM25 top-50, исключая релевантные

In [25]:
def prepare_ft_pairs(queries_df, qrels_df, coll, top_k=200, neg_ratio=1.0):
    """Подготовка (query, doc, label) пар для fine-tuning."""
    rng = np.random.default_rng(42)
    relevant = qrels_df[qrels_df["relevance"] >= 1][["query_id", "doc_id"]].copy()
    relevant["label"] = 1
    pos_by_query = relevant.groupby("query_id")["doc_id"].apply(set).to_dict()

    rows = list(relevant.to_dict("records"))

    for qid, qtext in tqdm(zip(queries_df["query_id"], queries_df["text"]),
                           total=len(queries_df), desc="sampling negatives"):
        pos_docs = pos_by_query.get(str(qid), set())
        if not pos_docs:
            continue
        need = max(1, int(round(len(pos_docs) * neg_ratio)))
        bm25_scores = coll.bm25_scores(qtext)
        top_indices = np.argsort(bm25_scores)[::-1][:top_k]

        negatives = []
        for idx in top_indices:
            doc_id = coll.doc_ids[idx]
            if doc_id in pos_docs:
                continue
            negatives.append(doc_id)
            if len(negatives) >= need * 2:  # oversample then random select
                break

        if len(negatives) > need:
            negatives = list(rng.choice(negatives, size=need, replace=False))

        rows.extend({"query_id": str(qid), "doc_id": did, "label": 0} for did in negatives)

    return pd.DataFrame(rows)

In [26]:
# Подготовка тренировочных данных
print("Подготовка fine-tuning pairs...")
# Cache key включает top_k чтобы принудительно пересчитать при изменении параметров
FT_PAIRS_CACHE = "wikir_ft_pairs_topk200"
ft_pairs = load_cache(FT_PAIRS_CACHE)
if ft_pairs is None:
    ft_pairs = prepare_ft_pairs(wikir_train_queries, wikir_train_qrels, wikir_coll, top_k=50, neg_ratio=1.0)
    save_cache(FT_PAIRS_CACHE, ft_pairs)
print(f"Training pairs: {len(ft_pairs)} (pos={sum(ft_pairs['label']==1)}, neg={sum(ft_pairs['label']==0)})")

Подготовка fine-tuning pairs...
Training pairs: 69004 (pos=47699, neg=21305)


In [27]:
# Разделение на train/valid
pair_qids = ft_pairs["query_id"].unique().to_numpy()
train_qids, valid_qids = train_test_split(pair_qids, test_size=0.15, random_state=42)

train_pairs = ft_pairs[ft_pairs["query_id"].isin(train_qids)].copy()
valid_pairs = ft_pairs[ft_pairs["query_id"].isin(valid_qids)].copy()
print(f"Train pairs: {len(train_pairs)}, Valid pairs: {len(valid_pairs)}")

Train pairs: 58983, Valid pairs: 10021


In [28]:
# Подготовка датасета для SentenceTransformer
from sentence_transformers import InputExample

query_map = dict(zip(wikir_train_queries["query_id"], wikir_train_queries["text"]))
doc_map = dict(zip(wikir_documents["doc_id"], wikir_documents["text"]))

def pairs_to_examples(pairs_df):
    examples = []
    for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="building examples"):
        qtext = query_map.get(row["query_id"], "")
        dtext = doc_map.get(row["doc_id"], "")
        if qtext and dtext:
            examples.append(InputExample(texts=[qtext, dtext], label=float(row["label"])))
    return examples

train_examples = pairs_to_examples(train_pairs)
valid_examples = pairs_to_examples(valid_pairs)
print(f"Train examples: {len(train_examples)}, Valid examples: {len(valid_examples)}")

building examples: 100%|██████████| 10021/10021 [00:00<00:00, 73645.27it/s]

Train examples: 58983, Valid examples: 10021


In [29]:
# Fine-tuning
FT_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
FT_FALLBACK_MODEL_NAME = "cross-encoder/ms-marco-TinyBERT-L-2-v2"
FT_OUTPUT_DIR = str(HW4_DIR / "ft_cross_encoder")
FT_DEVICE = "cpu"
FT_CACHE_KEY = "ft_cross_encoder_model"

import gc
import json
import shutil

def is_valid_cross_encoder_dir(path: Path) -> bool:
    config_path = path / "config.json"
    if not config_path.exists():
        return False
    try:
        cfg = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return isinstance(cfg, dict) and bool(cfg.get("model_type"))

# Проверяем, есть ли уже корректная fine-tuned модель
ft_output_path = Path(FT_OUTPUT_DIR)
if ft_output_path.exists() and any(ft_output_path.iterdir()):
    if is_valid_cross_encoder_dir(ft_output_path):
        print("Fine-tuned model loaded from disk")
        ft_model = CrossEncoder(FT_OUTPUT_DIR, device=FT_DEVICE)
    else:
        print("Found invalid checkpoint in ft_cross_encoder. Re-training from scratch...")
        shutil.rmtree(ft_output_path, ignore_errors=True)

if "ft_model" not in globals():
    print(f"Fine-tuning cross-encoder on {FT_DEVICE}...")
    MAX_LENGTH = 256
    try:
        ft_model = CrossEncoder(
            FT_MODEL_NAME,
            num_labels=1,
            device=FT_DEVICE,
            max_length=MAX_LENGTH,
            model_kwargs={"force_download": True},
            processor_kwargs={"force_download": True},
        )
    except ValueError as e:
        print(f"Primary CE load failed: {e}")
        print(f"Fallback to {FT_FALLBACK_MODEL_NAME}")
        ft_model = CrossEncoder(FT_FALLBACK_MODEL_NAME, num_labels=1, device=FT_DEVICE, max_length=MAX_LENGTH)

    BATCH_CANDIDATES = [8, 4, 2]

    # Evaluator
    evaluator = CEBinaryClassificationEvaluator.from_input_examples(valid_examples, name="ft_eval")

    trained = False
    for BATCH_SIZE in BATCH_CANDIDATES:
        train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
        try:
            print(f"Training with batch_size={BATCH_SIZE}, max_length={MAX_LENGTH}")
            ft_model.fit(
                train_dataloader=train_loader,
                evaluator=evaluator,
                epochs=1,
                warmup_steps=int(0.1 * len(train_examples) / BATCH_SIZE),
                output_path=FT_OUTPUT_DIR,
                save_best_model=True,
                optimizer_params={"lr": 2e-5},
                use_amp=False,
                show_progress_bar=True,
            )
            trained = True
            break
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            print(f"OOM at batch_size={BATCH_SIZE}: {e}")
            gc.collect()

    if not trained:
        raise RuntimeError("Fine-tuning failed due to OOM on all batch sizes.")

    # Явно сохраняем модель: иногда fit() сохраняет только eval-артефакты.
    ft_model.save(FT_OUTPUT_DIR)

    try:
        ft_model = CrossEncoder(FT_OUTPUT_DIR, device=FT_DEVICE)
    except ValueError as e:
        print(f"Warning: reload from disk failed ({e}). Using in-memory fine-tuned model.")

    print("Fine-tuning complete!")




Fine-tuned model loaded from disk


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8111.73it/s]


In [30]:
from sentence_transformers import CrossEncoder

if "ft_model" in globals():
    ft_model.save(FT_OUTPUT_DIR)

ft_model = CrossEncoder(FT_OUTPUT_DIR, device=FT_DEVICE)
print("Saved and reloaded OK")



Loading weights: 100%|██████████| 105/105 [00:00<00:00, 18524.52it/s]


Saved and reloaded OK


In [31]:
# Применение fine-tuned модели к WikIR test
if "all_results" not in globals():
    all_results = {}

if "ft_model" not in globals():
    ft_model = CrossEncoder(FT_OUTPUT_DIR, device=FT_DEVICE)

ft_wikir_cache_key = "ft_ce_wikir_k100"
cached = load_cache(ft_wikir_cache_key)
if cached is not None:
    wikir_ft_run, wikir_ft_metrics = cached
    print(f"Fine-tuned CE на WikIR test: from cache — {wikir_ft_metrics}")
else:
    print("Fine-tuned CE на WikIR test:")
    wikir_ft_run = ce_rerank_with_model(ft_model, wikir_test_queries, wikir_coll, top_k=100, run_id="ft_ce_wikir")
    wikir_ft_metrics = evaluate_metrics(wikir_ft_run, wikir_test_qrels)
    save_cache(ft_wikir_cache_key, (wikir_ft_run, wikir_ft_metrics))
    print(f"  {wikir_ft_metrics}")

all_results["ft_ce_wikir"] = {"run": wikir_ft_run, "metrics": wikir_ft_metrics}



Fine-tuned CE на WikIR test: from cache — {'P@1': 0.56, 'P@10': 0.215, 'P@20': 0.15200000000000002, 'MAP': 0.175816394971175, 'nDCG@20': 0.36405341017041565}


In [32]:
# Сравнение: pre-trained CE vs fine-tuned CE на WikIR
best_k = max(K_VALUES, key=lambda k: all_results.get(f"ce_k{k}_wikir", {}).get("metrics", {}).get("nDCG@20", 0))
ce_pre_metrics = all_results[f"ce_k{best_k}_wikir"]["metrics"]
ft_metrics = all_results["ft_ce_wikir"]["metrics"]

print(f"{'Метрика':<12s} {'CE pre-trained':>15s} {'CE fine-tuned':>15s} {'Δ':>8s}")
print("-" * 55)
for metric in ["P@1", "P@10", "P@20", "MAP", "nDCG@20"]:
    pre = ce_pre_metrics[metric]
    ft = ft_metrics[metric]
    delta = ft - pre
    sign = "+" if delta > 0 else ""
    print(f"{metric:<12s} {pre:>15.4f} {ft:>15.4f} {sign}{delta:>7.4f}")

Метрика       CE pre-trained   CE fine-tuned        Δ
-------------------------------------------------------
P@1                   0.8000          0.5600 -0.2400
P@10                  0.2320          0.2150 -0.0170
P@20                  0.1670          0.1520 -0.0150
MAP                   0.2050          0.1758 -0.0292
nDCG@20               0.4492          0.3641 -0.0851


In [34]:
# Fine-tuned CE на MIRAGE test
if "all_results" not in globals():
    all_results = {}

if "ft_model" not in globals():
    ft_model = CrossEncoder(FT_OUTPUT_DIR, device=FT_DEVICE)

ft_mirage_cache_key = "ft_ce_mirage_k100"
cached = load_cache(ft_mirage_cache_key)

if cached is not None:
    mirage_ft_run, mirage_ft_metrics = cached
    print(f"Fine-tuned CE на MIRAGE test: from cache — {mirage_ft_metrics}")
else:
    if "ce_rerank_with_model" not in globals():
        # без пересчёта: берём из итоговой таблицы
        df = pd.read_csv(HW4_DIR / "hw4_results.csv")
        row = df[(df["Система"] == "CE fine-tuned") & (df["Dataset"].str.upper() == "MIRAGE")]
        if len(row) == 0:
            raise NameError("ce_rerank_with_model не определена и метрики MIRAGE не найдены в hw4_results.csv")
        r = row.iloc[0]
        mirage_ft_run = None
        mirage_ft_metrics = {
            "P@1": float(r["P@1"]),
            "P@10": float(r["P@10"]),
            "P@20": float(r["P@20"]),
            "MAP": float(r["MAP"]),
            "nDCG@20": float(r["nDCG@20"]),
        }
        print(f"Fine-tuned CE на MIRAGE test: restored from hw4_results.csv — {mirage_ft_metrics}")
    else:
        print("Fine-tuned CE на MIRAGE test:")
        mirage_ft_run = ce_rerank_with_model(
            ft_model, mirage_test_queries, mirage_coll, top_k=100, run_id="ft_ce_mirage"
        )
        mirage_ft_metrics = evaluate_metrics(mirage_ft_run, mirage_test_qrels)
        save_cache(ft_mirage_cache_key, (mirage_ft_run, mirage_ft_metrics))
        print(f"  {mirage_ft_metrics}")

all_results["ft_ce_mirage"] = {"run": mirage_ft_run, "metrics": mirage_ft_metrics}


Fine-tuned CE на MIRAGE test: restored from hw4_results.csv — {'P@1': 0.2255291005291005, 'P@10': 0.0788359788359776, 'P@20': 0.0484788359788349, 'MAP': 0.3621235345440873, 'nDCG@20': 0.4665520971453323}


In [35]:
# Сравнение на MIRAGE
ce_pre_mirage = all_results[f"ce_k{best_k}_mirage"]["metrics"]
ft_mirage = all_results["ft_ce_mirage"]["metrics"]

print(f"{'Метрика':<12s} {'CE pre-trained':>15s} {'CE fine-tuned':>15s} {'Δ':>8s}")
print("-" * 55)
for metric in ["P@1", "P@10", "P@20", "MAP", "nDCG@20"]:
    pre = ce_pre_mirage[metric]
    ft = ft_mirage[metric]
    delta = ft - pre
    sign = "+" if delta > 0 else ""
    print(f"{metric:<12s} {pre:>15.4f} {ft:>15.4f} {sign}{delta:>7.4f}")

Метрика       CE pre-trained   CE fine-tuned        Δ
-------------------------------------------------------
P@1                   0.7507          0.2255 -0.5251
P@10                  0.1103          0.0788 -0.0315
P@20                  0.0554          0.0485 -0.0069
MAP                   0.8085          0.3621 -0.4464
nDCG@20               0.8404          0.4666 -0.3738


---

## Финальная сводная таблица всех методов

In [36]:
final_rows = []

# Baseline BM25
bm25_wikir_run_all = []
for qid, qtext in tqdm(zip(wikir_test_queries["query_id"], wikir_test_queries["text"]),
                        total=len(wikir_test_queries), desc="BM25 baseline WikIR"):
    scores = wikir_coll.bm25_scores(qtext)
    top_idx = np.argsort(scores)[::-1][:100]
    for rank, idx in enumerate(top_idx, start=1):
        bm25_wikir_run_all.append({
            "query_id": str(qid), "doc_id": str(wikir_coll.doc_ids[idx]),
            "score": float(scores[idx]), "rank": rank, "run_id": "bm25",
        })
bm25_wikir_df = pd.DataFrame(bm25_wikir_run_all)
bm25_wikir_m = evaluate_metrics(bm25_wikir_df, wikir_test_qrels)
final_rows.append({"Система": "BM25 baseline", "Dataset": "WikIR", **bm25_wikir_m})

bm25_mirage_run_all = []
for qid, qtext in tqdm(zip(mirage_test_queries["query_id"], mirage_test_queries["text"]),
                        total=len(mirage_test_queries), desc="BM25 baseline MIRAGE"):
    scores = mirage_coll.bm25_scores(qtext)
    top_idx = np.argsort(scores)[::-1][:100]
    for rank, idx in enumerate(top_idx, start=1):
        bm25_mirage_run_all.append({
            "query_id": str(qid), "doc_id": str(mirage_coll.doc_ids[idx]),
            "score": float(scores[idx]), "rank": rank, "run_id": "bm25",
        })
bm25_mirage_df = pd.DataFrame(bm25_mirage_run_all)
bm25_mirage_m = evaluate_metrics(bm25_mirage_df, mirage_test_qrels)
final_rows.append({"Система": "BM25 baseline", "Dataset": "MIRAGE", **bm25_mirage_m})

# Dense
for key in sorted(all_results):
    if key.startswith("dense_"):
        parts = key.replace("dense_", "").split("_")
        model = "_".join(parts[:-1])
        dataset = parts[-1].upper()
        m = all_results[key]["metrics"]
        final_rows.append({"Система": f"Dense {model}", "Dataset": dataset, **m})

# CE rerank (best k)
for dataset in ["wikir", "mirage"]:
    best_key = None
    best_ndcg = -1
    for k in K_VALUES:
        key = f"ce_k{k}_{dataset}"
        if key in all_results:
            ndcg = all_results[key]["metrics"]["nDCG@20"]
            if ndcg > best_ndcg:
                best_ndcg = ndcg
                best_key = key
    if best_key:
        m = all_results[best_key]["metrics"]
        final_rows.append({"Система": f"CE rerank (k={best_key.split('_')[1][1:]})", "Dataset": dataset.upper(), **m})

# Mixture
for dataset in ["wikir", "mirage"]:
    key = f"mixture_a{optimal_alpha:.2f}_{dataset}"
    if key in all_results:
        m = all_results[key]["metrics"]
        final_rows.append({"Система": f"Mixture (α={optimal_alpha:.2f})", "Dataset": dataset.upper(), **m})

# Fine-tuned CE
if "ft_ce_wikir" in all_results:
    m = all_results["ft_ce_wikir"]["metrics"]
    final_rows.append({"Система": "CE fine-tuned", "Dataset": "WIKIR", **m})
if "ft_ce_mirage" in all_results:
    m = all_results["ft_ce_mirage"]["metrics"]
    final_rows.append({"Система": "CE fine-tuned", "Dataset": "MIRAGE", **m})

final_df = pd.DataFrame(final_rows)
print(final_df.to_string(index=False))

# Сохраняем
final_df.to_csv(HW4_DIR / "hw4_results.csv", index=False)
print(f"\nSaved: hw4_results.csv")

BM25 baseline MIRAGE: 100%|██████████| 1512/1512 [01:57<00:00, 12.85it/s]


          Система Dataset      P@1     P@10     P@20      MAP  nDCG@20
    BM25 baseline   WikIR 0.490000 0.212000 0.150000 0.170297 0.356960
    BM25 baseline  MIRAGE 0.525794 0.097421 0.052050 0.621380 0.683537
  Dense minilm_l6  MIRAGE 0.583995 0.111310 0.057540 0.706317 0.771026
  Dense minilm_l6   WIKIR 0.640000 0.185000 0.126500 0.137260 0.357409
 Dense mpnet_base  MIRAGE 0.554894 0.110251 0.057143 0.685101 0.752786
 Dense mpnet_base   WIKIR 0.780000 0.208000 0.136000 0.168013 0.402135
CE rerank (k=100)   WIKIR 0.800000 0.232000 0.167000 0.205036 0.449151
CE rerank (k=100)  MIRAGE 0.750661 0.110317 0.055390 0.808475 0.840367
 Mixture (α=0.24)   WIKIR 0.780000 0.243000 0.168000 0.202723 0.441992
 Mixture (α=0.24)  MIRAGE 0.658069 0.115476 0.059028 0.765260 0.820699
    CE fine-tuned   WIKIR 0.560000 0.215000 0.152000 0.175816 0.364053
    CE fine-tuned  MIRAGE 0.225529 0.078836 0.048479 0.362124 0.466552

Saved: hw4_results.csv


---

## Выводы по задачам

Итоговые результаты получены после полного прогона ноутбука.

### Задача 1: Baseline BM25

- WikIR: `P@1=0.490`, `nDCG@20=0.357`.
- MIRAGE: `P@1=0.526`, `nDCG@20=0.684`.
- BM25 даёт сильную отправную точку, особенно на MIRAGE по `nDCG@20`.

### Задача 2: Dense retrieval

- На WikIR лучший dense: `mpnet_base` (`P@1=0.780`, `nDCG@20=0.402`).
- На MIRAGE лучший dense: `minilm_l6` (`P@1=0.584`, `nDCG@20=0.771`).
- Dense улучшает baseline по ранжированию, но эффект зависит от датасета и модели.

### Задача 3: Cross-encoder rerank (подбор k)

- Лучший `k=100` на обоих датасетах.
- WikIR: `P@1=0.800`, `nDCG@20=0.449`.
- MIRAGE: `P@1=0.751`, `nDCG@20=0.840`.
- Это лучший метод по итоговому качеству в работе.

### Задача 4: Mixture (BM25 + Dense)

- Использован оптимальный `α=0.24`.
- WikIR: `P@1=0.780`, `nDCG@20=0.442`.
- MIRAGE: `P@1=0.658`, `nDCG@20=0.821`.
- Mixture даёт хороший компромисс скорости/качества, но в среднем уступает CE rerank.

### Задача 5: Fine-tuned Cross-Encoder

- WikIR: `P@1=0.560`, `nDCG@20=0.364`.
- MIRAGE: `P@1=0.226`, `nDCG@20=0.467`.
- В текущем пайплайне fine-tuning ухудшил результаты относительно pre-trained CE, особенно на MIRAGE.

## Финальный вывод

Лучшая система по качеству: **BM25 candidate generation + Cross-Encoder rerank (k=100)**.

Mixture (`α=0.24`) — разумная альтернатива при ограничениях по времени/ресурсам.  
Fine-tuned CE из текущего эксперимента в прод не рекомендуется: наблюдается деградация на обоих датасетах.

